In [4]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive


def run(cmd, cwd=None):
    print("Running:", " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code:
        raise RuntimeError(
            f"Command failed with exit code {code}: "
            + " ".join(map(str, cmd))
        )


drive.mount("/content/drive")

WORKDIR = Path("/content/drive/MyDrive/Research/SSM_World_Models")
R2DREAMER_DIR = WORKDIR / "r2dreamer"
TDMPC2_DIR = WORKDIR / "tdmpc2"
DATA_DIR = WORKDIR / "data" / "dmc_expert"

R2DREAMER_REPO = "https://github.com/hanswalker/r2dreamer.git"
R2DREAMER_BRANCH = "mamba3-rssm-experiment"
TDMPC2_REPO = "https://github.com/nicklashansen/tdmpc2.git"

WORKDIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Colab starts with only this notebook. Clone the R2Dreamer repo that contains
# scripts/collect_dmc_expert_data.py, or update the Drive clone from the same
# GitHub source even if its default origin points somewhere else.
if not R2DREAMER_DIR.exists():
    run([
        "git",
        "clone",
        "--branch",
        R2DREAMER_BRANCH,
        R2DREAMER_REPO,
        str(R2DREAMER_DIR),
    ])
else:
    remotes = subprocess.run(
        ["git", "remote"],
        cwd=R2DREAMER_DIR,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.split()
    if "notebook-source" in remotes:
        run(["git", "remote", "set-url", "notebook-source", R2DREAMER_REPO], cwd=R2DREAMER_DIR)
    else:
        run(["git", "remote", "add", "notebook-source", R2DREAMER_REPO], cwd=R2DREAMER_DIR)
    run(["git", "fetch", "notebook-source", R2DREAMER_BRANCH], cwd=R2DREAMER_DIR)
    run(["git", "checkout", "-B", R2DREAMER_BRANCH, f"notebook-source/{R2DREAMER_BRANCH}"], cwd=R2DREAMER_DIR)

# TD-MPC2 is the expert policy repo. The collector imports its model code and
# downloads its released checkpoints from Hugging Face.
if not TDMPC2_DIR.exists():
    run(["git", "clone", TDMPC2_REPO, str(TDMPC2_DIR)])
else:
    run(["git", "pull", "--ff-only"], cwd=TDMPC2_DIR)

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "torch",
    "numpy==1.26.0",
    "pyyaml",
    "zarr<3",
    "huggingface_hub",
    "dm_control==1.0.28",
    "mujoco==3.3.0",
    "omegaconf",
    "hydra-core",
    "tensordict",
    "torchrl",
    "kornia",
    "termcolor",
    "tqdm",
    "pandas",
    "moviepy",
    "imageio",
    "imageio-ffmpeg",
    "h5py",
])

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

os.chdir(R2DREAMER_DIR)
if str(R2DREAMER_DIR) not in sys.path:
    sys.path.insert(0, str(R2DREAMER_DIR))

collector_candidates = [
    R2DREAMER_DIR / "scripts" / "collect_dmc_expert_data.py",
    R2DREAMER_DIR / "collect_dmc_expert_data.py",
]
COLLECTOR = next((path for path in collector_candidates if path.exists()), None)
if COLLECTOR is None:
    raise FileNotFoundError(
        "Missing collect_dmc_expert_data.py in the cloned R2Dreamer repo. "
        f"Checked: {collector_candidates}"
    )

R2DREAMER_COMMIT = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=R2DREAMER_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

import torch

print("R2Dreamer:", R2DREAMER_DIR)
print("R2Dreamer source:", R2DREAMER_REPO, R2DREAMER_BRANCH, R2DREAMER_COMMIT)
print("TD-MPC2:", TDMPC2_DIR)
print("Data:", DATA_DIR)
print("Collector:", COLLECTOR)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "TD-MPC2 requires CUDA. In Colab, choose Runtime > Change runtime type > GPU, "
        "then rerun the notebook."
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running: git remote add notebook-source https://github.com/hanswalker/r2dreamer.git
Running: git fetch notebook-source mamba3-rssm-experiment
From https://github.com/hanswalker/r2dreamer
 * branch            mamba3-rssm-experiment -> FETCH_HEAD
 * [new branch]      mamba3-rssm-experiment -> notebook-source/mamba3-rssm-experiment
Running: git checkout -B mamba3-rssm-experiment notebook-source/mamba3-rssm-experiment
Reset branch 'mamba3-rssm-experiment'
M	runs/atari.sh
M	runs/crafter.sh
M	runs/dmc.sh
M	runs/dmc_subtle.sh
M	runs/isaaclab.sh
M	runs/memorymaze.sh
M	runs/metaworld.sh
Branch 'mamba3-rssm-experiment' set up to track remote branch 'mamba3-rssm-experiment' from 'notebook-source'.
Your branch is up to date with 'origin/mamba3-rssm-experiment'.
Running: git pull --ff-only
Already up to date.
Running: /usr/bin/python3 -m pip install -q torch numpy==1.26.0

In [5]:
CONFIG_PATH = DATA_DIR / "collect_dmc_expert_smoke.yaml"
CONFIG_PATH.write_text(
    f"""
tdmpc2_root: {TDMPC2_DIR}
output_dir: {DATA_DIR}

tasks:
  - cartpole/swingup

num_episodes: 2
checkpoint_seed: 1
seed: 1

action_repeat: 2
max_episode_steps: 500

save_images: false
image_size: 64

resume: true
""".strip()
    + "\n",
    encoding="utf-8",
)

print("Config:", CONFIG_PATH)
print(CONFIG_PATH.read_text())
run([sys.executable, str(COLLECTOR), "--help"])
run([sys.executable, str(COLLECTOR), "--config", str(CONFIG_PATH)])


Config: /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert/collect_dmc_expert_smoke.yaml
tdmpc2_root: /content/drive/MyDrive/Research/SSM_World_Models/tdmpc2
output_dir: /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert

tasks:
  - cartpole/swingup

num_episodes: 2
checkpoint_seed: 1
seed: 1

action_repeat: 2
max_episode_steps: 500

save_images: false
image_size: 64

resume: true

Running: /usr/bin/python3 /content/drive/MyDrive/Research/SSM_World_Models/r2dreamer/scripts/collect_dmc_expert_data.py --help
usage: collect_dmc_expert_data.py [-h] [--config CONFIG]

Collect DMC expert rollouts with TD-MPC2 checkpoints.

options:
  -h, --help       show this help message and exit
  --config CONFIG  YAML config file for collection settings.
Running: /usr/bin/python3 /content/drive/MyDrive/Research/SSM_World_Models/r2dreamer/scripts/collect_dmc_expert_data.py --config /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert/collect_dmc_expert_smoke.yaml
/u

RuntimeError: Command failed with exit code 1: /usr/bin/python3 /content/drive/MyDrive/Research/SSM_World_Models/r2dreamer/scripts/collect_dmc_expert_data.py --config /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert/collect_dmc_expert_smoke.yaml